# ⚙️ Notebook 1: WMX R2 Infrastructure & Engine Startup

This notebook initializes the **WMX3 Motion Engine** and activates the EtherCAT communication network to establish the foundational infrastructure for robot control.

### 🎯 Objective & Scope
* **Focus**: Executes the sequential startup process, from launching background servers to allocating the motion engine, starting EtherCAT communication, and loading parameters.
* **Note**: This notebook does *not* perform actual motor actuation (Servo On and motion are covered in Notebooks 2 and 3).

### 🔄 Initialization Sequence
1. **Step 1: Start ROS 2 Backend Servers**: Launch the communication nodes to process client requests.
2. **Step 2: Create WMX3 Device**: Allocate the core motion engine instance in memory.
3. **Step 3: Start EtherCAT Communication**: Initiate master-slave network communication to connect the hardware.
4. **Step 4: Load XML Parameters**: Upload physical configuration parameters (e.g., gear ratios, limits) to the engine.

## Step 1: Start ROS 2 Backend Servers

Before executing any Python client commands, you must launch the background ROS 2 server nodes to receive and process your requests.

### 🛠️ Terminal Execution
1. Open a new terminal by clicking the Jupyter logo (top-right) ➡ `New` ➡ `Terminal`.
2. Copy and run the command below. Once the system stabilizes, return to this notebook and proceed.
3. Once the terminal output stabilizes and shows node registration, return to this notebook and proceed to the cells below.

```bash
sudo --preserve-env=PATH \
     --preserve-env=AMENT_PREFIX_PATH \
     --preserve-env=COLCON_PREFIX_PATH \
     --preserve-env=PYTHONPATH \
     --preserve-env=LD_LIBRARY_PATH \
     --preserve-env=ROS_DISTRO \
     --preserve-env=ROS_VERSION \
     --preserve-env=ROS_PYTHON_VERSION \
     --preserve-env=ROS_DOMAIN_ID \
     --preserve-env=RMW_IMPLEMENTATION \
     bash -c "source /opt/ros/${ROS_DISTRO}/setup.bash && source $HOME/workspaces/movensys_ws/install/setup.bash && \00
     ros2 launch wmx_r2_package wmx_r2_general_nodes.launch.py"  
```

#### 🔌 Jupyter ↔ ROS 2 Client Connection (common to every notebook)
Once the backend nodes are running in a terminal, this Jupyter kernel needs an `rclpy` client to talk to them. The cell below creates one node using the **`WmxClient`** helper defined in `wmx_utils.py` — every notebook from 01 through 05 reuses this class's `wmx.call(service_type, service_name, request)` method for its service calls, so it's worth understanding the pattern here once.

* `node_name` is set differently in each notebook (`wmx_client_01`, `wmx_client_02`, ...). Using the same node name in multiple kernels at once causes a name collision in the ROS 2 graph.
* `wmx.axis_list` is the list of axis indices this exercise controls; it defaults to `[0, 1]` in `wmx_utils.py`. Adjust it there to match the number of axes on your hardware.

In [ ]:
import rclpy
from std_srvs.srv import SetBool
from wmx_r2_message.srv import SetEngine, LoadWmxParams, SetAxis
from wmx_utils import WmxClient

if not rclpy.ok():
    rclpy.init()

wmx = WmxClient(node_name='wmx_client_01')
print(f"✅ WMX Client Ready for Notebook 01. Target Axes: {wmx.axis_list}")

## Step 2: Create WMX3 Device Interface

Request the backend server to instantiate the WMX3 motion controller in memory, building the core software framework.

In [ ]:
req = SetEngine.Request()
req.data = True
req.path = '/opt/wmx3/'
req.name = 'my_device'

res = wmx.call(SetEngine, '/wmx/engine/set_device', req)
if res.success:
    print(f"✅ {res.message}")
else:
    print(f"❌ [ERROR] {res.message}")

In [ ]:
# ==============================================================================
# ⚠️ [OPTIONAL] WMX Device Shutdown Code
# Only run this cell when you are completely finished with your entire practice!
# ==============================================================================
"""
RUN_SHUTDOWN = False

if RUN_SHUTDOWN:
    req = SetEngine.Request()
    req.data = False
    req.path = '/opt/wmx3/'
    req.name = 'my_device'

    res = wmx.call(SetEngine, '/wmx/engine/set_device', req)
    if res.success:
        print(f"✅ [SUCCESS] {res.message}")
    else:
        print(f"❌ [ERROR] {res.message}")
"""

## Step 3: Start EtherCAT Communication

Start the EtherCAT master network to establish a real-time data exchange state (OP State) with the connected motor drives (slaves).

In [ ]:
req = SetBool.Request()
req.data = True

res = wmx.call(SetBool, '/wmx/engine/set_comm', req)
if res.success:
    print(f"✅ [SUCCESS] {res.message}")
else:
    print(f"❌ [ERROR] {res.message}")

## Step 4: Load WMX Parameter XML File

Apply the physical configuration parameters (gear ratios, travel limits, encoder profiles, etc.) required for your specific hardware to the motion engine.

In [ ]:
import os

xml_path = os.path.expanduser('~/workspaces/movensys_ws/install/wmx_r2_package/share/wmx_r2_package/config/diffbot_wmx_parameters.xml')

req = LoadWmxParams.Request()
req.file_path = xml_path

res = wmx.call(LoadWmxParams, '/wmx/params/load', req)
if res.success:
    print(f"✅ XML parameters injected: \n{xml_path}")
else:
    print(f"❌ [ERROR] {res.message}")